# 1. 環境の準備

In [1]:
!pip install -U huggingface_hub psutil -q

In [3]:
# llama-cpp-python を最適化付きで最新のソースから再ビルドしてインストールする
!CMAKE_ARGS="-DGGML_BLAS=ON -DGGML_BLAS_VENDOR=OpenBLAS" \
pip install --upgrade --force-reinstall --no-cache-dir \
git+https://github.com/abetlen/llama-cpp-python.git

Looking in indexes: https://pypi.org/simple, https://www.piwheels.org/simple
  Cloning https://github.com/abetlen/llama-cpp-python.git to /tmp/pip-req-build-gp_3vrhc
  Running command git clone --filter=blob:none --quiet https://github.com/abetlen/llama-cpp-python.git /tmp/pip-req-build-gp_3vrhc
  Resolved https://github.com/abetlen/llama-cpp-python.git to commit 1b1a320de18c14c3915ba2df59eedb7c6e7cbe69
  Running command git submodule update --init --recursive -q
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 4.1 MB/s  0:00:034.1 MB/s eta 0:00:01:01
  Created wheel for llama_cpp_python: filename=llama_cpp_python-0.3.20-py3-none-linux_aarch64.whl size=12667959 sha256=f070a4ee881e46a2e3f8ad62728cab0ff1715ad9a5ab3a3a2172a90b5f1934fb
  Stored in directory: /tmp/pip-ephem-wheel-cache-ynirehni/wheels/39/68/e8/c1f8392f3d2961e620d2757f09ae90167945a211

In [13]:
import os
import time
import psutil
from llama_cpp import Llama
from huggingface_hub import list_repo_files
from huggingface_hub import hf_hub_download

# 2. 関数定義
## 1. モデルダウンロード

In [6]:
MODEL_DIR = "/home/pi/models"

def download_gguf(repo_id, filename):
    os.makedirs(MODEL_DIR, exist_ok=True)

    filepath = os.path.join(MODEL_DIR, filename)

    if os.path.exists(filepath):
        return filepath

    return hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        local_dir=MODEL_DIR,
        local_dir_use_symlinks=False
    )

##  2. 推論コード（共通テンプレート）

In [7]:
def run_inference(gguf_path, prompt, n_ctx=2048, n_threads=4,max_tokens=100,):
    print(f"\n=== MODEL ===\n{gguf_path}")

    print("=== モデルロード ===")
    llm = Llama(
        model_path=gguf_path,
        n_ctx=n_ctx,
        n_threads=n_threads,
        n_gpu_layers=0,   # Raspberry Pi / CPU前提
        verbose=False,    # ログ抑制
    )
    
    start = time.perf_counter()
    print("=== 推論開始 ===")
    
    out = llm(
        prompt,
        max_tokens=max_tokens,
        temperature=0.1,
        top_p=0.9,
        stop=["<|system|>", "<|user|>"],
    )
    
    elapsed = time.perf_counter() - start
    response = out["choices"][0]["text"].strip()
    
    print("=== 推論完了 ===")
    print(response)
    print("\n--- STATS ---")
    print(f"推論時間: {elapsed:.2f} 秒")

In [8]:
import time
from llama_cpp import Llama

# =========================
# モデルロード
# =========================
def load_model(gguf_path, n_ctx=2048, n_threads=4):
    print(f"\n=== モデルロード: {gguf_path} ===")
    start = time.perf_counter()

    llm = Llama(
        model_path=gguf_path,
        n_ctx=n_ctx,
        n_threads=n_threads,
        n_gpu_layers=0,
        verbose=False,
    )

    elapsed = time.perf_counter() - start
    print(f"ロード時間: {elapsed:.2f} 秒")
    return llm


# =========================
# 推論実行
# =========================
def run_inference(llm, prompt, max_tokens=100):
    start = time.perf_counter()

    out = llm(
        prompt,
        max_tokens=max_tokens,
        temperature=0.1,
        top_p=0.9,
        stop=["<|system|>", "<|user|>"],
    )

    elapsed = time.perf_counter() - start
    response = out["choices"][0]["text"].strip()

    # トークン数を取得（usageがあれば）
    try:
        tokens = out["usage"]["completion_tokens"]
    except:
        tokens = len(response.split())

    tps = tokens / elapsed if elapsed > 0 else 0

    print("=== 推論完了 ===")
    print(response)
    print("\n--- STATS ---")
    print(f"時間: {elapsed:.2f} 秒")
    print(f"トークン数: {tokens}")
    print(f"処理速度: {tps:.2f} tokens/sec")

    return {
        "response": response,
        "time": elapsed,
        "tokens": tokens,
        "tokens_per_sec": tps,
    }

## 3. プロンプト

In [9]:
prompt = "Explain Raspberry Pi in simple terms."

In [10]:
prompt_instruct = """
<|system|>
You are a teacher for children.
<|user|>
Explain what a Raspberry Pi is in a way that elementary school kids can understand.
Requirements:
- Use no more than 2 sentences.
- Avoid technical jargon as much as possible.
<|assistant|>
"""

## 4. 推論

### 4-1. Phi-2
[https://huggingface.co/TheBloke/phi-2-GGUF](https://huggingface.co/TheBloke/phi-2-GGUF)  



In [8]:
repo_id = "TheBloke/phi-2-GGUF"
files = list_repo_files(repo_id)

for f in files:
    if f.endswith(".gguf"):
        print(f)

phi-2.Q2_K.gguf
phi-2.Q3_K_L.gguf
phi-2.Q3_K_M.gguf
phi-2.Q3_K_S.gguf
phi-2.Q4_0.gguf
phi-2.Q4_K_M.gguf
phi-2.Q4_K_S.gguf
phi-2.Q5_0.gguf
phi-2.Q5_K_M.gguf
phi-2.Q5_K_S.gguf
phi-2.Q6_K.gguf
phi-2.Q8_0.gguf


In [9]:
# モデルを取得
gguf_path = download_gguf("TheBloke/phi-2-GGUF", "phi-2.Q4_K_M.gguf")

# 推論実行
llm = load_model(gguf_path, n_ctx=1024, n_threads=4)
result = run_inference(llm, prompt)


=== モデルロード: /home/pi/models/phi-2.Q4_K_M.gguf ===


llama_context: n_ctx_per_seq (1024) < n_ctx_train (2048) -- the full capacity of the model will not be utilized


ロード時間: 20.58 秒
=== 推論完了 ===
Raspberry Pi is a small computer that can do many things. It is like a toy that can also do homework. You can use it to make games, robots, or even a weather station. It is very cheap and easy to use.

--- STATS ---
時間: 8.33 秒
トークン数: 49
処理速度: 5.88 tokens/sec


### 4-2. Phi-3 mini

In [10]:
repo_id = "bartowski/Phi-3.5-mini-instruct-GGUF"
files = list_repo_files(repo_id)

for f in files:
    if f.endswith(".gguf"):
        print(f)

Phi-3.5-mini-instruct-IQ2_M.gguf
Phi-3.5-mini-instruct-IQ3_M.gguf
Phi-3.5-mini-instruct-IQ3_XS.gguf
Phi-3.5-mini-instruct-IQ4_XS.gguf
Phi-3.5-mini-instruct-Q2_K.gguf
Phi-3.5-mini-instruct-Q2_K_L.gguf
Phi-3.5-mini-instruct-Q3_K_L.gguf
Phi-3.5-mini-instruct-Q3_K_M.gguf
Phi-3.5-mini-instruct-Q3_K_S.gguf
Phi-3.5-mini-instruct-Q3_K_XL.gguf
Phi-3.5-mini-instruct-Q4_0.gguf
Phi-3.5-mini-instruct-Q4_0_4_4.gguf
Phi-3.5-mini-instruct-Q4_0_4_8.gguf
Phi-3.5-mini-instruct-Q4_0_8_8.gguf
Phi-3.5-mini-instruct-Q4_K_L.gguf
Phi-3.5-mini-instruct-Q4_K_M.gguf
Phi-3.5-mini-instruct-Q4_K_S.gguf
Phi-3.5-mini-instruct-Q5_K_L.gguf
Phi-3.5-mini-instruct-Q5_K_M.gguf
Phi-3.5-mini-instruct-Q5_K_S.gguf
Phi-3.5-mini-instruct-Q6_K.gguf
Phi-3.5-mini-instruct-Q6_K_L.gguf
Phi-3.5-mini-instruct-Q8_0.gguf
Phi-3.5-mini-instruct-f32.gguf


In [11]:
# モデルを取得
gguf_path = download_gguf(
    "bartowski/Phi-3.5-mini-instruct-GGUF", 
    "Phi-3.5-mini-instruct-Q4_K_M.gguf"
)
# 推論実行
llm = load_model(gguf_path, n_ctx=1024, n_threads=4)
result = run_inference(llm, prompt)


=== モデルロード: /home/pi/models/Phi-3.5-mini-instruct-Q4_K_M.gguf ===


llama_context: n_ctx_per_seq (1024) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


ロード時間: 27.43 秒
=== 推論完了 ===
Raspberry Pi is a small, affordable computer that you can use for a variety of projects. It's like a tiny computer that fits in the palm of your hand. It's made by a company called Raspberry Pi Foundation, and it's designed to be easy to use and learn from, especially for people who are new to computers.

Here's a simple breakdown of what it is:

1. **Mini Computer**

--- STATS ---
時間: 24.38 秒
トークン数: 100
処理速度: 4.10 tokens/sec


In [12]:
# instruct用のプロンプトを使用
result = run_inference(llm, prompt_instruct)

=== 推論完了 ===
A Raspberry Pi is like a tiny computer that fits in your hand, and you can use it to play games, learn about computers, or even build your own projects like robots!

--- STATS ---
時間: 34.41 秒
トークン数: 100
処理速度: 2.91 tokens/sec


### 4-3. Qwen2.5

In [13]:
repo_id = "tensorblock/Qwen2.5-0.5B-GGUF"

files = list_repo_files(repo_id)

print("=== 全ファイル一覧 ===")
for f in files:
    print(f)

=== 全ファイル一覧 ===
.gitattributes
Qwen2.5-0.5B-Q2_K.gguf
Qwen2.5-0.5B-Q3_K_M.gguf
README.md


In [14]:
# モデルを取得
gguf_path = download_gguf(
    "tensorblock/Qwen2.5-0.5B-GGUF",
    "Qwen2.5-0.5B-Q3_K_M.gguf"
)
# 推論実行
llm = load_model(gguf_path, n_ctx=1024, n_threads=4)
result = run_inference(llm, prompt)


=== モデルロード: /home/pi/models/Qwen2.5-0.5B-Q3_K_M.gguf ===


llama_context: n_ctx_per_seq (1024) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


ロード時間: 4.50 秒
=== 推論完了 ===
Raspberry Pi is a small, low-cost computer that can be used for a variety of tasks, including controlling a camera, sending data to a server, and running a web browser. It is powered by a Raspberry Pi B or B+ board, which is a small, low-power, and low-cost computer. Raspberry Pi is designed to be easy to use and can be controlled from a web browser or through a USB connection.

--- STATS ---
時間: 2.98 秒
トークン数: 85
処理速度: 28.57 tokens/sec


### 4-5. Granite 4.0 h-tiny
[https://huggingface.co/ibm-granite/granite-4.0-h-tiny-GGUF](hhttps://huggingface.co/ibm-granite/granite-4.0-h-tiny-GGUF)

In [15]:
repo_id = "ibm-granite/granite-4.0-h-tiny-GGUF"
files = list_repo_files(repo_id)

for f in files:
    print(f)

.gitattributes
README.md
granite-4.0-h-tiny-Q2_K.gguf
granite-4.0-h-tiny-Q3_K_L.gguf
granite-4.0-h-tiny-Q3_K_M.gguf
granite-4.0-h-tiny-Q3_K_S.gguf
granite-4.0-h-tiny-Q4_0.gguf
granite-4.0-h-tiny-Q4_1.gguf
granite-4.0-h-tiny-Q4_K_M.gguf
granite-4.0-h-tiny-Q4_K_S.gguf
granite-4.0-h-tiny-Q5_0.gguf
granite-4.0-h-tiny-Q5_1.gguf
granite-4.0-h-tiny-Q5_K_M.gguf
granite-4.0-h-tiny-Q5_K_S.gguf
granite-4.0-h-tiny-Q6_K.gguf
granite-4.0-h-tiny-Q8_0.gguf
granite-4.0-h-tiny-f16.gguf


In [16]:
# モデルを取得
gguf_path = download_gguf(
    "ibm-granite/granite-4.0-h-tiny-GGUF",
    "granite-4.0-h-tiny-Q4_K_M.gguf"
)
# 推論実行
llm = load_model(gguf_path, n_ctx=1024, n_threads=4)
result = run_inference(llm, prompt)


=== モデルロード: /home/pi/models/granite-4.0-h-tiny-Q4_K_M.gguf ===


llama_context: n_ctx_per_seq (1024) < n_ctx_train (1048576) -- the full capacity of the model will not be utilized
llama_kv_cache_unified: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 512


ロード時間: 49.36 秒
=== 推論完了 ===
**

**Answer:**  
The Raspberry Pi is a small, affordable computer that can be used for learning programming and creating projects. It's like a tiny computer that can run programs, connect to the internet, and control devices, making it perfect for beginners to explore technology and electronics.

---

**What is the difference between a laptop and a desktop?**

**Answer:**  
A laptop is a portable computer that combines the display, keyboard, and touchpad into one unit, allowing you to use it anywhere

--- STATS ---
時間: 18.96 秒
トークン数: 100
処理速度: 5.27 tokens/sec


### 4-6. Granite 4.0 h-350m
[https://huggingface.co/ibm-granite/granite-4.0-h-tiny-GGUF](hhttps://huggingface.co/ibm-granite/granite-4.0-h-tiny-GGUF)

In [17]:
repo_id = "ibm-granite/granite-4.0-h-350m-GGUF"
files = list_repo_files(repo_id)
for f in files:
    print(f)

.gitattributes
README.md
granite-4.0-h-350m-F16.gguf
granite-4.0-h-350m-Q2_K.gguf
granite-4.0-h-350m-Q3_K_L.gguf
granite-4.0-h-350m-Q3_K_M.gguf
granite-4.0-h-350m-Q3_K_S.gguf
granite-4.0-h-350m-Q4_0.gguf
granite-4.0-h-350m-Q4_1.gguf
granite-4.0-h-350m-Q4_K_M.gguf
granite-4.0-h-350m-Q4_K_S.gguf
granite-4.0-h-350m-Q5_0.gguf
granite-4.0-h-350m-Q5_1.gguf
granite-4.0-h-350m-Q5_K_M.gguf
granite-4.0-h-350m-Q5_K_S.gguf
granite-4.0-h-350m-Q6_K.gguf
granite-4.0-h-350m-Q8_0.gguf
granite-4.0-h-350m-bf16.gguf


In [18]:
# モデルを取得
gguf_path = download_gguf(
    "ibm-granite/granite-4.0-h-350m-GGUF",
    "granite-4.0-h-350m-Q8_0.gguf"
)
# 推論実行
llm = load_model(gguf_path, n_ctx=1024, n_threads=4)
result = run_inference(llm, prompt)


=== モデルロード: /home/pi/models/granite-4.0-h-350m-Q8_0.gguf ===


llama_context: n_ctx_per_seq (1024) < n_ctx_train (1048576) -- the full capacity of the model will not be utilized
llama_kv_cache_unified: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 256


ロード時間: 4.61 秒
=== 推論完了 ===
What are the key features of Raspberry Pi?

--- STATS ---
時間: 0.90 秒
トークン数: 9
処理速度: 10.03 tokens/sec


### 4-8. Bonsai-8B.gguf

[https://huggingface.co/prism-ml/Bonsai-8B-gguf](https://huggingface.co/prism-ml/Bonsai-8B-gguf)

In [11]:
from huggingface_hub import list_repo_files
repo_id = "prism-ml/Bonsai-8B-gguf"
files = list_repo_files(repo_id)
for f in files:
    print(f)

.eval_results/gsm8k.yaml
.gitattributes
Bonsai-8B-Q1_0.gguf
Bonsai-8B.gguf
LICENSE
NOTICE.txt
README.md
assets/bonsai-logo.svg
assets/energy_8B.png
assets/frontier.png
assets/frontier.svg
assets/intel_density_8B.png
assets/speeds_8B.png


In [14]:
# Bonsai GGUF を取得
gguf_path = download_gguf(
    "prism-ml/Bonsai-8B-gguf",
    "Bonsai-8B.gguf"
)

In [15]:
# 推論実行
llm = load_model(gguf_path, n_ctx=4096, n_threads=4)
result = run_inference(llm, prompt)


=== モデルロード: /home/pi/models/Bonsai-8B.gguf ===


llama_context: n_ctx_seq (4096) < n_ctx_train (65536) -- the full capacity of the model will not be utilized


ロード時間: 4.07 秒
=== 推論完了 ===
What are its main uses?

Okay, so I need to explain Raspberry Pi in simple terms. Let me start by recalling what I know. Raspberry Pi is a single-board computer, right? It's like a computer that's smaller than a regular PC. It's made by a company called Raspberry Pi, which is based in the UK. The main thing is that it's affordable and has a lot of features for its size.

First, what's a single-board computer? Well, it's a

--- STATS ---
時間: 69.77 秒
トークン数: 100
処理速度: 1.43 tokens/sec
{'response': "What are its main uses?\n\nOkay, so I need to explain Raspberry Pi in simple terms. Let me start by recalling what I know. Raspberry Pi is a single-board computer, right? It's like a computer that's smaller than a regular PC. It's made by a company called Raspberry Pi, which is based in the UK. The main thing is that it's affordable and has a lot of features for its size.\n\nFirst, what's a single-board computer? Well, it's a", 'time': 69.77444865300004, 'tokens': 100,